# 01.06 · Data wrangling: pandas vs polars (vs DuckDB)

**Chapter:** Python & Math for AI · **Milestone:** M1

## Goals
- [ ] Load CSV / JSONL / Parquet; filter, derive, group, join
- [ ] Do the same work in pandas, polars and SQL (DuckDB) and know when each fits

## Resources
- [pandas user guide](https://pandas.pydata.org/docs/user_guide/) — free
- [Polars user guide](https://docs.pola.rs/) — free
- [DuckDB Python docs](https://duckdb.org/docs/stable/clients/python/overview) — free

| Tool | Strengths | Use when |
|---|---|---|
| **pandas** | everywhere, every tutorial, integrates with everything | default; small-medium data; plotting |
| **polars** | much faster, multi-threaded, lazy queries, strict types | big files, performance-sensitive pipelines |
| **DuckDB** | SQL over files/dataframes, in-process | you think in SQL; querying Parquet directly |

The dataset: a synthetic **LLM call log** — the kind of data you'll analyse constantly as an AI engineer
(which model is slow? which feature costs most?).

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import polars as pl

from aieng import DATA

rng = np.random.default_rng(0)
N = 200_000
models = np.array(["gemini-flash", "gpt-oss-120b", "llama-3.3-70b", "qwen3-1.7b"])
features = np.array(["chat", "rag", "summarise", "agent"])
m = rng.choice(models, N, p=[0.4, 0.25, 0.2, 0.15])

calls = pd.DataFrame({
    "ts": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 30 * 24 * 3600, N), unit="s"),
    "model": m,
    "feature": rng.choice(features, N),
    "input_tokens": rng.integers(50, 8000, N),
    "output_tokens": rng.integers(10, 1500, N),
    "latency_ms": (rng.lognormal(6.5, 0.5, N) * np.where(m == "qwen3-1.7b", 2.0, 1.0)).round(),
    "error": rng.random(N) < 0.02,
})
# Illustrative prices in $ per 1M tokens (input, output). Real prices: check provider pricing pages.
prices = pd.DataFrame({
    "model": models,
    "usd_per_m_in": [0.30, 0.15, 0.60, 0.0],
    "usd_per_m_out": [2.50, 0.60, 0.80, 0.0],
})
calls.head()

## 1. File formats
- **CSV**: human-readable, no types, slow, big.
- **JSONL**: one JSON object per line; standard for prompts, eval sets, fine-tuning data, traces.
- **Parquet**: columnar, typed, compressed, fast. Use for anything large.

In [ ]:
calls.to_csv(DATA / "calls.csv", index=False)
calls.to_parquet(DATA / "calls.parquet")
calls.head(1000).to_json(DATA / "calls.jsonl", orient="records", lines=True, date_format="iso")
for ext in ["csv", "parquet", "jsonl"]:
    print(ext, f"{(DATA / f'calls.{ext}').stat().st_size / 1e6:.1f} MB")
print(json.loads((DATA / "calls.jsonl").read_text().splitlines()[0]))

## 2. pandas: filter, derive, join, group

In [ ]:
def report_pandas(calls, prices):
    df = calls[~calls["error"]].merge(prices, on="model")  # drop failed calls, join price table
    df["cost_usd"] = (df["input_tokens"] * df["usd_per_m_in"] + df["output_tokens"] * df["usd_per_m_out"]) / 1e6
    df["tokens_per_s"] = df["output_tokens"] / (df["latency_ms"] / 1000)
    return (
        df.groupby(["model", "feature"])
        .agg(calls=("cost_usd", "size"), total_cost=("cost_usd", "sum"),
             p50_latency=("latency_ms", "median"), p95_latency=("latency_ms", lambda s: s.quantile(0.95)))
        .sort_values("total_cost", ascending=False)
    )

report_pandas(calls, prices).head(8)

## 3. polars: same report
Polars uses **expressions** (`pl.col(...)`) instead of indexing. `.lazy()` builds a query plan and optimises it before running.

In [ ]:
calls_pl, prices_pl = pl.from_pandas(calls), pl.from_pandas(prices)

def report_polars(calls, prices):
    return (
        calls.lazy()
        .filter(~pl.col("error"))
        .join(prices.lazy(), on="model")
        .with_columns(
            cost_usd=(pl.col("input_tokens") * pl.col("usd_per_m_in") + pl.col("output_tokens") * pl.col("usd_per_m_out")) / 1e6,
        )
        .group_by("model", "feature")
        .agg(
            calls=pl.len(), total_cost=pl.col("cost_usd").sum(),
            p50_latency=pl.col("latency_ms").median(), p95_latency=pl.col("latency_ms").quantile(0.95),
        )
        .sort("total_cost", descending=True)
        .collect()
    )

report_polars(calls_pl, prices_pl).head(8)

## 4. Time series: daily cost per feature

In [ ]:
daily = (
    calls.merge(prices, on="model")
    .assign(cost=lambda d: (d.input_tokens * d.usd_per_m_in + d.output_tokens * d.usd_per_m_out) / 1e6)
    .set_index("ts").groupby("feature")["cost"].resample("D").sum().unstack(0)
)
daily.plot(figsize=(10, 3), title="Daily cost by feature ($)");

## 5. Speed comparison

In [ ]:
def timeit(fn, *args, n=3):
    start = time.perf_counter()
    for _ in range(n):
        fn(*args)
    return (time.perf_counter() - start) / n

print(f"pandas: {timeit(report_pandas, calls, prices) * 1000:.0f} ms | polars: {timeit(report_polars, calls_pl, prices_pl) * 1000:.0f} ms")

## Exercises
1. Which model has the highest **error rate** per feature? (both libraries)
2. Add a `tokens_per_s` column and find the fastest model at p50.
3. `uv add duckdb`, then write the main report as one SQL query: `duckdb.sql("SELECT ... FROM 'data/calls.parquet' ...")`.
   DuckDB can query the Parquet file directly without loading it first.
4. Generate N = 5,000,000 rows and redo the speed comparison. Where does pandas start hurting?

In [ ]:
# your code here